We have Demeter ICE VLF power spectrum data in Binary format. Here we are converting a single Binary DAT file to ASCII format.

In [1]:
import numpy as np
import os 
import struct
from datetime import datetime, date, timedelta 
import pandas as pd


In [5]:

RECL = 8510
# the Input file is a DAT file. 
input_file = "F:\\1132-ICE_VLF\\1132_19-04(5)\\DMT_N1_1132_149901_20070423_234407_20070424_001756.DAT"



In [6]:
Datetime =[]
Lt = []
Al = []
Lat = []
Long = []
dfArray = []
frq =[]

In [7]:


with open(input_file, 'rb') as f:
    piece = f.read(RECL)
    
    counter = 0  # count of the number of occurrences of RELC
    while piece:
        
        counter += 1
        datcds = [int.from_bytes(piece[:4], byteorder='big'), int.from_bytes(piece[4:8], byteorder='big')]
        day=np.mod(datcds[0],16777216)
        msec=datcds[1]
        record_date = date(1950, 1, 1) + timedelta(days=int(day))
        #print("Block 1")
        #print(f"mpd = {day}, msec = {msec}, date = {record_date}")
        i = 8; delta = 2 * 7
        time = struct.unpack('>'+'h'*7, piece[i:i+delta])
        #print(f"time = {time}")
        dt = datetime(year=int(time[0]), month=int(time[1]), day=int(time[2]), hour=int(time[3]),minute=int(time[4]), second=int(time[5]),microsecond=int(time[6])*1000)
        date_time = dt.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3]
        start_time = datetime.strptime(date_time,'%Y-%m-%d %H:%M:%S.%f')
        #print(start_time)

        i = i + delta; delta = 2 * 2
      
        orb, sorb = struct.unpack('>hh', piece[i:i + delta])
        
        i = i + delta; delta = 8
       
        tms = "".join([s.decode() for s in struct.unpack('>' +'s'*8, piece[i:i+delta])])
       
        i = i + delta; delta = 4
        soft_cal = struct.unpack('>bbbb', piece[i:i + delta])
        
        i = i + delta; delta = 4*4
        orbp = struct.unpack('>ffff', piece[i:i+delta])
        #print(f"Geocentric_lat = {orbp[0]:.4f}, Geocentric_long = {orbp[1]:.4f}")
        #print(f"Altitude = {orbp[2]:.4f}, local_time = {orbp[3]:.4f}")
        Geocentric_lat = f"{orbp[0]:.4f}"
        Geocentric_long = f"{orbp[1]:.4f}"
        local_time = f"{orbp[3]:.4f}"
        Altitude = f"{orbp[2]:.3f}"
        i += delta; delta = 4*15
        geop = struct.unpack('>'+'f'*15, piece[i:i+delta])
        
      
        i += delta; delta = 4*3
        solp = struct.unpack('>'+'f'*3, piece[i:i+delta])
        
        i = i + delta; delta = 2
        vers = struct.unpack('>bb', piece[i:i + delta])
        
        i += delta; delta = 4*18
        attp = struct.unpack('>'+'f'*18, piece[i:i+delta])
        
        i = i + delta; delta = 2
        qi = struct.unpack('>h', piece[i:i + delta])[0]
        
        i = i + delta; delta = 2
        vers = struct.unpack('>bb', piece[i:i + delta])
        
        i = i + delta; delta = 21
        type = "".join([s.decode() for s in struct.unpack('>' +'s'*21, piece[i:i+delta])])
        
        i = i + delta; delta = 32
        hk = struct.unpack('>'+'c'*32, piece[i:i + delta])
        HKs = "".join([el.hex() for el in hk])
        
        i = i + delta; delta = 9
        coord = "".join([s.decode() for s in struct.unpack('>' +'s'*9, piece[i:i+delta])])
        
        i = i + delta; delta = 3
        name = "".join([s.decode() for s in struct.unpack('>' +'s'*3, piece[i:i+delta])])
        
        i = i + delta; delta = 16
        unit = "".join([s.decode('latin-1') for s in struct.unpack('>' +'s'*16, piece[i:i+delta])])
        
        i = i + delta; delta = 1
        nbsp = struct.unpack('>'+'b'*1, piece[i:i + delta])[0]
        #print(f"Number of consecutice spectra = {nbsp}")
        i = i + delta; delta = 2
        nbf = struct.unpack('>h', piece[i:i + delta])[0]
        #print(f"Number of spectrum frequencies  = {nbf}")
        i += delta; delta = 4
        dt = struct.unpack('>f', piece[i:i+delta])[0]
        #print(f"Time duration of one data array = {dt:.4f}")
        i += delta; delta = 4
        freq = struct.unpack('>f', piece[i:i+delta])[0]
        #print(f"Frequency resolution = {freq:.4f}")
        i += delta; delta = 4*2
        frange = struct.unpack('>ff', piece[i:i+delta])[0]
        #print(f"Frequency range = {frange:.4f}")
        i += delta; delta = 2 * 7
        idatsp = struct.unpack('>'+'h'*7, piece[i:i+delta])
        #print(f"time = {idatsp}")
        i += delta
        sp = struct.unpack('>2048f', piece[i:])
        sp_0 = sp[:1024]
        sp_1 = sp[1024:]
        sp0=list(sp_0)
        sp1=list(sp_1)
        data1 = {'spectrum_0': sp0, 'spectrum_1': sp1}
        # Create a Pandas data frame from the dictionary
        df = pd.DataFrame(data1)
        dfArray.append(df) 
        #sp = struct.unpack('>2048f', piece[i:])
        for i in range(nbf):
            time_delta = timedelta(seconds=i * 1/freq)
            current_time = start_time + time_delta

            current_time_str = current_time.strftime("%Y-%m-%d %H:%M:%S.%f")
            Datetime.append(current_time_str)
            Lt.append(local_time)
            Al.append(Altitude)
            Lat.append(Geocentric_lat)
            Long.append(Geocentric_long)
            frq.append(freq)
        piece = f.read(RECL)
     
 
print(f'the count of repeatation of relc : {counter}')
        



the count of repeatation of relc : 495


In [8]:
Master_SP =pd.concat(dfArray,axis=0) #Spectrum DATA
Master = Master_sp.reset_index(drop = True)  
OP = pd.DataFrame({'Geocentric_lat':Lat,'Geocentric_long':Long,'Altitude': Al,"Local_time":Lt,'frequency':frq})           
DT = pd.DataFrame(Datetime,columns=['Datetime'])
data = pd.concat([DT,OP,Master], ignore_index = False, axis=1)

#Save the dataframe to a Pickle file

data.to_pickle("F:\\EQ-27-02-2010\\DMT_N1_1132_149901_20070423_234407_20070424_001756.pkl")
